# 📘 Streaming & Real-Time Data Processing
## Databricks Notebook — Structured Streaming with Delta Lake

**What you'll learn:**
- Batch vs micro-batch vs real-time processing
- Structured Streaming: readStream, writeStream, triggers
- Streaming from Delta tables (append + CDF)
- Windowed aggregations (tumbling, sliding)
- Watermarks for late data handling
- Stream-static and stream-stream joins
- Auto Loader patterns, Kafka concepts
- Production streaming patterns

**Key rules:**
- Use `trigger(availableNow=True)` for batch-style streaming in notebooks
- Delta tables as both source and sink (exactly-once guarantees)
- Community Edition supports Structured Streaming

**Prerequisites:** Notebooks 01-23 (iot_manufacturing_dw)

---

---
# 📗 Section 1: Streaming Concepts

## Batch vs Micro-Batch vs Real-Time

| Mode | Latency | Use Case | Databricks |
|---|---|---|---|
| Batch | Minutes-Hours | Daily reports, ETL | Regular Spark jobs |
| Micro-batch | Seconds-Minutes | Near-real-time dashboards | Structured Streaming |
| Real-time | Milliseconds | Fraud detection, alerts | Continuous trigger |

## Key Concepts

- **Exactly-once:** Each record processed exactly once (no duplicates, no loss)
- **Event time:** When the event actually happened (from source)
- **Processing time:** When Spark processes it (wall clock)
- **Watermark:** How long to wait for late data before closing a window
- **Checkpoint:** Saves progress so streaming can resume after failure

## When Streaming vs Batch?

| Use Streaming | Use Batch |
|---|---|
| Data arrives continuously | Data arrives in scheduled dumps |
| Need < 5 min latency | Hourly/daily is fine |
| Event-driven processing | Report generation |
| Anomaly detection | Historical analysis |

## Batch vs Streaming — When to Use Each

### The Fundamental Trade-off

```
BATCH PROCESSING:                        STREAMING PROCESSING:
┌─────────────────────────┐             ┌─────────────────────────┐
│ Process data in chunks  │             │ Process data as it      │
│ (hourly, daily, weekly) │             │ arrives (seconds)       │
│                         │             │                         │
│ ✅ Simple to build      │             │ ✅ Low latency           │
│ ✅ Easy to debug        │             │ ✅ Real-time insights    │
│ ✅ Lower cost           │             │ ✅ Immediate action      │
│ ❌ High latency         │             │ ❌ More complex          │
│ ❌ Stale data           │             │ ❌ Higher cost           │
└─────────────────────────┘             └─────────────────────────┘
```

### Decision Framework

| Use Case | Latency Needed | Use |
|----------|---------------|-----|
| Daily revenue report | Hours | Batch |
| Fraud detection | < 1 second | Streaming |
| Inventory dashboard | 15 minutes | Micro-batch |
| Real-time pricing | < 100ms | Streaming |
| ML model training | Hours | Batch |
| Anomaly alerting | < 30 seconds | Streaming |
| Monthly billing | Days | Batch |
| Live leaderboard | < 5 seconds | Streaming |

### Structured Streaming — Unified API

Spark Structured Streaming treats a stream as an **unbounded table**:

```
Batch DataFrame:                         Streaming DataFrame:
┌────────────────────┐                  ┌────────────────────┐
│ id │ amount │ date │                  │ id │ amount │ date │ ← New rows
│  1 │  100   │ 3/15 │                  │  1 │  100   │ 3/15 │   keep
│  2 │  200   │ 3/15 │                  │  2 │  200   │ 3/15 │   arriving
│  3 │  300   │ 3/16 │                  │  3 │  300   │ 3/16 │   forever
└────────────────────┘                  │  4 │  400   │ 3/17 │ ←
                                        │  5 │  500   │ 3/17 │ ←
                                        └────────────────────┘

Same API! df.filter(), df.groupBy(), df.join() work on BOTH.
The difference: streaming DataFrames have a .writeStream instead of .write
```

In [ ]:
# Demonstrate the unified batch/streaming API
from pyspark.sql.functions import col, sum as spark_sum, count, avg

print("🔄 UNIFIED BATCH/STREAMING API")
print("=" * 60)

# The SAME transformation logic works for both batch and streaming!
def compute_daily_revenue(df):
    """This function works for BOTH batch and streaming DataFrames."""
    return (df
        .filter(col("status").isin("completed", "shipped"))
        .groupBy("order_date")
        .agg(
            count("*").alias("total_orders"),
            spark_sum("total_amount").alias("total_revenue"),
            avg("total_amount").alias("avg_order_value")
        ))

# Batch mode
batch_df = spark.table("techmart_dw.orders")
batch_result = compute_daily_revenue(batch_df)
print("\n📊 Batch mode result:")
batch_result.orderBy("order_date").show(5)

# Streaming mode would use the SAME function:
print("\n📡 Streaming mode (conceptual):")
print("""
   stream_df = spark.readStream.format("delta").table("bronze.orders")
   stream_result = compute_daily_revenue(stream_df)  # SAME function!
   
   stream_result.writeStream
       .format("delta")
       .outputMode("complete")
       .option("checkpointLocation", "/checkpoints/daily_revenue")
       .trigger(processingTime="5 minutes")
       .toTable("gold.daily_revenue")
""")
print("💡 Key Insight: Write your transformation logic ONCE.")
print("   It works for both batch (testing) and streaming (production).")


---
# 📗 Section 2: Structured Streaming Fundamentals

In [ ]:
# Rate source: generates data for testing (no external dependencies)
# Creates rows with (timestamp, value) at specified rate

rate_stream = (spark.readStream
    .format("rate")
    .option("rowsPerSecond", 100)
    .option("numPartitions", 2)
    .load()
)

print(f"Schema: {rate_stream.schema}")
print(f"Is streaming: {rate_stream.isStreaming}")

In [ ]:
# Write streaming data to Delta (trigger availableNow = process all available then stop)
from pyspark.sql.functions import *

# Transform the rate stream
transformed = (rate_stream
    .withColumn("sensor_id", (col("value") % 10 + 1).cast("int"))
    .withColumn("reading", (col("value") * 0.1 + 20).cast("double"))
    .withColumn("event_date", to_date("timestamp"))
)

# Write with trigger(availableNow=True) — processes available data then stops
query = (transformed.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "/tmp/streaming_demo/checkpoint")
    .trigger(availableNow=True)
    .toTable("iot_manufacturing_dw.streaming_demo")
)

query.awaitTermination()
print(f"✅ Streaming demo: {spark.table('iot_manufacturing_dw.streaming_demo').count()} rows written")

In [ ]:
# Output modes explained:
# - append: only new rows (default, most common)
# - complete: entire result table (for aggregations)
# - update: only changed rows (for aggregations with watermark)

# Triggers explained:
# - trigger(availableNow=True): process all available, then stop (RECOMMENDED for notebooks)
# - trigger(once=True): legacy version of availableNow
# - trigger(processingTime="10 seconds"): micro-batch every 10s (production)
# - continuous: true real-time (experimental)

print("Output modes: append (new rows), complete (all rows), update (changed rows)")
print("Triggers: availableNow (notebook), processingTime (production)")

## Triggers — Controlling When Streaming Runs

Triggers determine HOW OFTEN Spark processes new data:

### Trigger Types

```python
# 1. Fixed interval (most common for near-real-time)
.trigger(processingTime="30 seconds")  # Run every 30 seconds
.trigger(processingTime="5 minutes")   # Run every 5 minutes

# 2. Available Now (batch mode — process all, then stop)
.trigger(availableNow=True)

# 3. Continuous (experimental — very low latency, ~1ms)
.trigger(continuous="1 second")

# 4. Default (no trigger) — process as fast as possible
# No .trigger() call — runs micro-batches back-to-back
```

### Trigger Comparison

| Trigger | Latency | Cost | Use For |
|---------|---------|------|---------|
| `processingTime="1 minute"` | ~1 minute | Medium | Dashboards, alerts |
| `processingTime="5 minutes"` | ~5 minutes | Lower | Near-real-time |
| `availableNow=True` | Batch | Lowest | Scheduled jobs |
| No trigger | Seconds | Highest | Real-time |
| `continuous` | Milliseconds | Highest | Ultra-low latency |

## Output Modes — What Gets Written

| Mode | What Gets Written | Use For |
|------|------------------|---------|
| `append` | Only NEW rows | Append-only sources (logs, events) |
| `complete` | ENTIRE result table | Aggregations (GROUP BY) |
| `update` | Only CHANGED rows | Aggregations with updates |

```python
# append: new events only
.outputMode("append")   # ← Use for: raw events, no aggregation

# complete: full aggregation result
.outputMode("complete") # ← Use for: GROUP BY (rewrites entire result)

# update: only changed aggregation rows
.outputMode("update")   # ← Use for: GROUP BY with Delta sink
```

In [ ]:
# Output modes demonstration
from pyspark.sql.functions import col, count, sum as spark_sum, window

print("📊 OUTPUT MODES — Understanding the Difference")
print("=" * 60)

# Simulate streaming data arriving in batches
batches = [
    # Batch 1
    [("NYC", 100), ("LA", 200), ("NYC", 150)],
    # Batch 2 (new data arrives)
    [("NYC", 300), ("Chicago", 100)],
    # Batch 3
    [("LA", 250), ("Chicago", 200)],
]

# Simulate what each output mode would write
print("\nSimulating 3 micro-batches of city revenue data:")
print("Batch 1: NYC=100, LA=200, NYC=150")
print("Batch 2: NYC=300, Chicago=100")
print("Batch 3: LA=250, Chicago=200")

running_totals = {}

print("\n" + "="*60)
print("APPEND mode (only new rows):")
print("  Batch 1: NYC=100, LA=200, NYC=150  ← All new")
print("  Batch 2: NYC=300, Chicago=100       ← All new")
print("  Batch 3: LA=250, Chicago=200        ← All new")
print("  Result: 8 rows total (duplicates for same city!)")
print("  ⚠️ Not suitable for aggregations!")

print("\n" + "="*60)
print("COMPLETE mode (full aggregation result each time):")
for i, batch in enumerate(batches):
    for city, amount in batch:
        running_totals[city] = running_totals.get(city, 0) + amount
    print(f"  Batch {i+1}: {dict(sorted(running_totals.items()))}")
print("  Result: Full table rewritten each batch")
print("  ✅ Correct for aggregations, but expensive for large tables")

print("\n" + "="*60)
print("UPDATE mode (only changed rows):")
prev_totals = {}
for i, batch in enumerate(batches):
    changed = {}
    for city, amount in batch:
        old = running_totals.get(city, 0)
        running_totals[city] = old + amount
        changed[city] = running_totals[city]
    print(f"  Batch {i+1}: Only changed: {dict(sorted(changed.items()))}")
print("  ✅ Efficient: only writes rows that changed")


---
# 📗 Section 3: Streaming from Delta Tables

In [ ]:
# Read Delta table as a stream (only new rows since last checkpoint)
spark.sql("USE iot_manufacturing_dw")

# Delta table as streaming source (append-only)
sensor_stream = (spark.readStream
    .format("delta")
    .table("sensor_readings")
)

# Process: add anomaly flag
from pyspark.sql.functions import *
processed = (sensor_stream
    .filter("quality_flag = 'valid'")
    .withColumn("is_high_value", col("value") > 80)
    .withColumn("_processed_at", current_timestamp())
)

# Write to silver table
query = (processed.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "/tmp/streaming_silver/checkpoint")
    .trigger(availableNow=True)
    .toTable("iot_manufacturing_dw.streaming_silver_readings")
)
query.awaitTermination()
print(f"✅ Streaming silver: {spark.table('iot_manufacturing_dw.streaming_silver_readings').count():,} rows")

---
# 📗 Section 4: Windowed Aggregations

## Window Types

```
TUMBLING (fixed, non-overlapping):
|---5min---|---5min---|---5min---|

SLIDING (fixed, overlapping):
|---10min--|
     |---10min--|
          |---10min--|

SESSION (gap-based):
|--events--| gap |--events--|
```

In [ ]:
# Tumbling window aggregation (5-minute windows)
from pyspark.sql.functions import *

sensor_stream = spark.readStream.format("delta").table("sensor_readings")

# 5-minute tumbling window: avg, min, max per sensor
windowed = (sensor_stream
    .withWatermark("reading_timestamp", "10 minutes")  # Allow 10 min late data
    .groupBy(
        window("reading_timestamp", "5 minutes"),  # Tumbling window
        "sensor_id"
    )
    .agg(
        count("*").alias("reading_count"),
        round(avg("value"), 2).alias("avg_value"),
        round(min("value"), 2).alias("min_value"),
        round(max("value"), 2).alias("max_value")
    )
)

# Write windowed aggregates
query = (windowed.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "/tmp/streaming_windowed/checkpoint")
    .trigger(availableNow=True)
    .toTable("iot_manufacturing_dw.streaming_windowed_metrics")
)
query.awaitTermination()
print(f"✅ Windowed metrics: {spark.table('iot_manufacturing_dw.streaming_windowed_metrics').count():,} rows")

---
# 📗 Section 5: Watermarks & Late Data

**Watermark** = "How long to wait for late data before finalizing a window"

```python
.withWatermark("event_time", "10 minutes")
```

This means: data arriving more than 10 minutes late will be dropped.
Data within 10 minutes of the watermark will still be included.

In [ ]:
# ============================================
# 🎯 YOUR TURN: Streaming with Watermark
# ============================================
# 1. Read sensor_readings as a stream
# 2. Set watermark of 15 minutes on reading_timestamp
# 3. Create 10-minute tumbling windows
# 4. Aggregate: count, avg(value) per window per sensor_id
# 5. Write to a Delta table with trigger(availableNow=True)
#
# Write your code below:


In [ ]:
# ============================================
# ✅ SOLUTION
# ============================================
from pyspark.sql.functions import *

stream = spark.readStream.format("delta").table("iot_manufacturing_dw.sensor_readings")

result = (stream
    .withWatermark("reading_timestamp", "15 minutes")
    .groupBy(window("reading_timestamp", "10 minutes"), "sensor_id")
    .agg(count("*").alias("count"), round(avg("value"), 2).alias("avg_val"))
)

q = (result.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "/tmp/streaming_practice/cp")
    .trigger(availableNow=True)
    .toTable("iot_manufacturing_dw.practice_windowed")
)
q.awaitTermination()
print(f"✅ Practice windowed: {spark.table('iot_manufacturing_dw.practice_windowed').count():,} rows")

## Watermarks — Deep Dive

### The Late Data Problem

In distributed systems, events arrive OUT OF ORDER:

```
Real event times:  10:00  10:01  10:02  10:03  10:04  10:05
Arrival at Spark:  10:00  10:01  10:05  10:02  10:03  10:06
                                  ↑       ↑
                            LATE! (3 min)  LATE! (1 min)
```

Without watermarks, Spark would wait forever for late data.
With watermarks, Spark says: "I'll accept data up to X minutes late, then close the window."

### Watermark Mechanics

```python
df.withWatermark("event_time", "10 minutes")
  .groupBy(window("event_time", "5 minutes"))
  .agg(count("*"))
```

```
Watermark = max(event_time seen so far) - 10 minutes

If latest event is at 10:30:
  Watermark = 10:30 - 10 minutes = 10:20
  
  Windows ending BEFORE 10:20 are CLOSED (no more late data accepted)
  Windows ending AFTER 10:20 are OPEN (still accepting data)
```

### Watermark + Window Interaction

```
Events:  ●=on-time  ○=late

Time:    10:00  10:05  10:10  10:15  10:20  10:25  10:30
         ●      ●      ●      ●      ●      ●      ●
                              ○(late, 10:05 event arrives at 10:15)
                                     ○(late, 10:08 event arrives at 10:20)
                                            ✗(too late, 10:05 event at 10:25)

Window [10:00-10:10]:
  Closes when watermark passes 10:10
  Watermark = max_event - 10min
  If max_event = 10:20, watermark = 10:10 → window JUST closed
  Late events for this window arriving after 10:20 are DROPPED
```

### Choosing Watermark Duration

| Data Source | Typical Latency | Recommended Watermark |
|-------------|----------------|----------------------|
| Kafka (same datacenter) | < 1 second | 30 seconds - 2 minutes |
| Kafka (cross-region) | 1-5 seconds | 5-10 minutes |
| Mobile app events | 1-30 minutes | 30-60 minutes |
| IoT sensors (poor connectivity) | Minutes to hours | 2-4 hours |
| Batch files | N/A | Use batch, not streaming |

In [ ]:
# Watermark simulation
from datetime import datetime, timedelta

class WatermarkSimulator:
    """Simulates Spark watermark behavior."""
    
    def __init__(self, watermark_delay_minutes):
        self.watermark_delay = timedelta(minutes=watermark_delay_minutes)
        self.max_event_time = None
        self.watermark = None
        self.windows = {}  # window_start → {count, closed}
        self.dropped_events = 0
    
    def process_event(self, event_time_str, value, window_size_minutes=5):
        event_time = datetime.strptime(event_time_str, "%H:%M")
        
        # Update max event time and watermark
        if self.max_event_time is None or event_time > self.max_event_time:
            self.max_event_time = event_time
            self.watermark = self.max_event_time - self.watermark_delay
        
        # Determine which window this event belongs to
        window_size = timedelta(minutes=window_size_minutes)
        window_start = event_time - timedelta(
            minutes=event_time.minute % window_size_minutes,
            seconds=event_time.second
        )
        window_end = window_start + window_size
        
        # Check if window is already closed
        if self.watermark and window_end <= self.watermark:
            self.dropped_events += 1
            return f"DROPPED (window {window_start.strftime('%H:%M')}-{window_end.strftime('%H:%M')} closed)"
        
        # Add to window
        key = window_start.strftime("%H:%M")
        if key not in self.windows:
            self.windows[key] = {"count": 0, "closed": False}
        self.windows[key]["count"] += 1
        
        watermark_str = self.watermark.strftime("%H:%M") if self.watermark else "N/A"
        return f"ACCEPTED → window {key} (watermark: {watermark_str})"
    
    def show_windows(self):
        print("\n📊 Window Results:")
        for window_start, data in sorted(self.windows.items()):
            print(f"   Window {window_start}: {data['count']} events")


# Demo: 10-minute watermark, 5-minute windows
sim = WatermarkSimulator(watermark_delay_minutes=10)

events = [
    ("10:00", 1), ("10:02", 2), ("10:04", 3),
    ("10:06", 4), ("10:08", 5), ("10:10", 6),
    ("10:03", 7),  # Late! 10:03 event arrives after 10:10
    ("10:15", 8), ("10:17", 9),
    ("10:04", 10), # Very late! 10:04 event arrives after 10:17
    ("10:20", 11),
    ("10:02", 12), # Too late! Window [10:00-10:05] is closed
]

print("🌊 WATERMARK SIMULATION (10-minute delay, 5-minute windows)")
print("=" * 70)
print(f"{'Event Time':<12} {'Arrival Order':<6} {'Result'}")
print("-" * 70)
for i, (event_time, value) in enumerate(events):
    result = sim.process_event(event_time, value)
    late_marker = " ← LATE" if i >= 6 and "ACCEPTED" in result else ""
    dropped_marker = " ← DROPPED" if "DROPPED" in result else ""
    print(f"{event_time:<12} {i+1:<6} {result}{late_marker}{dropped_marker}")

sim.show_windows()
print(f"\n   Total dropped events: {sim.dropped_events}")
print("\n💡 Key Insight:")
print("   Late events within the watermark window are ACCEPTED.")
print("   Events beyond the watermark are DROPPED (window already closed).")
print("   Larger watermark = more late data accepted, but higher memory usage.")


---
# 📗 Section 6: Streaming Joins

In [ ]:
# Stream-static join: enrich streaming data with dimension table
from pyspark.sql.functions import *

# Static dimension (loaded once)
sensors_dim = spark.table("iot_manufacturing_dw.sensors").select("sensor_id", "sensor_type", "machine_id", "location")

# Streaming source
sensor_stream = spark.readStream.format("delta").table("iot_manufacturing_dw.sensor_readings")

# Join stream with static dimension (no watermark needed)
enriched = (sensor_stream
    .join(sensors_dim, "sensor_id", "left")
    .select("reading_id", "sensor_id", "sensor_type", "machine_id", "location",
        "reading_timestamp", "value", "quality_flag")
)

# Write enriched stream
q = (enriched.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "/tmp/streaming_enriched/cp")
    .trigger(availableNow=True)
    .toTable("iot_manufacturing_dw.streaming_enriched_readings")
)
q.awaitTermination()
print(f"✅ Enriched stream: {spark.table('iot_manufacturing_dw.streaming_enriched_readings').count():,} rows")

---
# 📗 Section 7: Writing with foreachBatch

In [ ]:
# foreachBatch: custom logic per micro-batch (most flexible)
from pyspark.sql.functions import *

def process_batch(batch_df, batch_id):
    """Custom processing for each micro-batch."""
    if batch_df.count() == 0:
        return
    
    # Add batch metadata
    enriched = (batch_df
        .withColumn("_batch_id", lit(f"batch_{batch_id}"))
        .withColumn("_processed_at", current_timestamp())
    )
    
    # Write to Delta (merge for upsert, or append)
    enriched.write.format("delta").mode("append").saveAsTable("iot_manufacturing_dw.streaming_foreach_output")

# Use foreachBatch
stream = spark.readStream.format("delta").table("iot_manufacturing_dw.sensor_readings")

q = (stream
    .filter("quality_flag = 'valid'")
    .writeStream
    .foreachBatch(process_batch)
    .option("checkpointLocation", "/tmp/streaming_foreach/cp")
    .trigger(availableNow=True)
    .start()
)
q.awaitTermination()
print(f"✅ foreachBatch output: {spark.table('iot_manufacturing_dw.streaming_foreach_output').count():,} rows")

## foreachBatch — Custom Sink Logic

`foreachBatch` lets you apply arbitrary batch operations to each micro-batch:

```python
def process_batch(batch_df, batch_id):
    """Called for each micro-batch. batch_df is a regular DataFrame."""
    
    # You can do ANYTHING here — it's just a batch DataFrame!
    
    # 1. Write to multiple destinations
    batch_df.write.format("delta").mode("append").saveAsTable("silver.orders")
    batch_df.filter(col("amount") > 1000).write.format("delta").mode("append").saveAsTable("silver.high_value_orders")
    
    # 2. Apply MERGE (upsert) — not possible with regular streaming sinks
    batch_df.createOrReplaceTempView("batch_orders")
    spark.sql("""
        MERGE INTO silver.orders AS target
        USING batch_orders AS source
        ON target.order_id = source.order_id
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    
    # 3. Send alerts for anomalies
    anomalies = batch_df.filter(col("amount") > 10000)
    if anomalies.count() > 0:
        send_slack_alert(f"High-value orders detected: {anomalies.count()}")
    
    # 4. Write to external systems (REST API, database)
    batch_df.toPandas().to_sql("orders", engine, if_exists="append")

# Apply to stream
stream.writeStream.foreachBatch(process_batch).start()
```

### When to Use foreachBatch

| Use Case | Why foreachBatch |
|----------|-----------------|
| MERGE/Upsert | Regular streaming can't do MERGE |
| Multiple sinks | Write to 2+ destinations per batch |
| External systems | REST API, JDBC, custom sinks |
| Complex logic | Conditional writes, alerts |
| Deduplication | Apply ROW_NUMBER before writing |

In [ ]:
# foreachBatch with MERGE pattern
from pyspark.sql.functions import col, current_timestamp, row_number
from pyspark.sql.window import Window

print("🔀 foreachBatch — MERGE Pattern")
print("=" * 60)

# Setup: Create target table
spark.sql("DROP TABLE IF EXISTS techmart_dw.streaming_orders_target")
spark.sql("""
CREATE TABLE techmart_dw.streaming_orders_target (
    order_id INT,
    customer_id INT,
    total_amount DECIMAL(12,2),
    status STRING,
    _last_updated TIMESTAMP
) USING DELTA
""")

# Simulate what foreachBatch does with each micro-batch
def simulate_foreach_batch(batch_data, batch_id):
    """Simulates the foreachBatch function."""
    print(f"\n   Processing batch {batch_id}: {len(batch_data)} records")
    
    # Create DataFrame from batch data
    batch_df = spark.createDataFrame(batch_data, 
        ["order_id", "customer_id", "total_amount", "status"])
    
    # Deduplicate within batch (keep latest per order_id)
    window = Window.partitionBy("order_id").orderBy(col("total_amount").desc())
    deduped = (batch_df
        .withColumn("rn", row_number().over(window))
        .filter(col("rn") == 1)
        .drop("rn")
        .withColumn("_last_updated", current_timestamp()))
    
    # MERGE into target (upsert)
    deduped.createOrReplaceTempView(f"batch_{batch_id}")
    spark.sql(f"""
        MERGE INTO techmart_dw.streaming_orders_target AS target
        USING batch_{batch_id} AS source
        ON target.order_id = source.order_id
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    
    count = spark.table("techmart_dw.streaming_orders_target").count()
    print(f"   Target table now has {count} rows")


# Simulate 3 micro-batches
batches = [
    [(1, 42, 99.99, "pending"), (2, 17, 149.50, "pending"), (3, 55, 299.00, "pending")],
    [(1, 42, 99.99, "completed"), (4, 88, 450.00, "pending")],  # Order 1 updated
    [(2, 17, 149.50, "shipped"), (5, 33, 75.00, "pending")],    # Order 2 updated
]

for i, batch in enumerate(batches):
    simulate_foreach_batch(batch, i + 1)

print("\n📊 Final target table:")
spark.table("techmart_dw.streaming_orders_target").orderBy("order_id").show()
print("✅ MERGE correctly handled updates (order 1: pending→completed, order 2: pending→shipped)")


---
# 📗 Section 8: Kafka Concepts (Conceptual)

⚠️ Kafka requires external infrastructure — patterns shown conceptually.

```python
# Reading from Kafka (CONCEPTUAL)
kafka_stream = (spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "sensor-events")
    .option("startingOffsets", "latest")
    .load()
    .select(
        col("key").cast("string"),
        from_json(col("value").cast("string"), schema).alias("data")
    )
    .select("data.*")
)

# Writing to Kafka (CONCEPTUAL)
(output_df.writeStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("topic", "processed-events")
    .option("checkpointLocation", "/checkpoints/kafka-out")
    .start()
)
```

**Key Kafka concepts:**
- **Topic:** Named stream of records (like a table)
- **Partition:** Parallel unit within a topic
- **Consumer group:** Set of consumers sharing the work
- **Offset:** Position in the stream (like a bookmark)

## Kafka Deep Dive — The Streaming Backbone

### What is Kafka?

Apache Kafka is a **distributed event streaming platform** — the most common
source for Spark Structured Streaming in production.

```
PRODUCERS                    KAFKA                    CONSUMERS
┌──────────┐                ┌──────────────────┐     ┌──────────────┐
│ Mobile   │───────────────▶│ Topic: orders    │────▶│ Spark        │
│ App      │                │ Partition 0: ●●● │     │ Streaming    │
└──────────┘                │ Partition 1: ●●● │     └──────────────┘
┌──────────┐                │ Partition 2: ●●● │     ┌──────────────┐
│ Backend  │───────────────▶│                  │────▶│ Analytics    │
│ Service  │                │ Topic: payments  │     │ Service      │
└──────────┘                │ Partition 0: ●●● │     └──────────────┘
┌──────────┐                │ Partition 1: ●●● │     ┌──────────────┐
│ IoT      │───────────────▶│                  │────▶│ Fraud        │
│ Devices  │                └──────────────────┘     │ Detection    │
└──────────┘                                         └──────────────┘
```

### Key Kafka Concepts

| Concept | Description | Analogy |
|---------|-------------|---------|
| **Topic** | Named stream of events | A TV channel |
| **Partition** | Ordered, immutable log within a topic | A lane on a highway |
| **Offset** | Position of a message in a partition | Page number in a book |
| **Producer** | Writes messages to topics | A broadcaster |
| **Consumer** | Reads messages from topics | A viewer |
| **Consumer Group** | Multiple consumers sharing work | A team watching together |
| **Broker** | Kafka server | A TV tower |
| **Retention** | How long messages are kept | DVR recording duration |

### Reading from Kafka in Spark

```python
# Read from Kafka
kafka_df = (spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker1:9092,broker2:9092")
    .option("subscribe", "orders")                    # Single topic
    # OR: .option("subscribePattern", "orders.*")     # Pattern match
    # OR: .option("assign", '{"orders": [0,1,2]}')    # Specific partitions
    .option("startingOffsets", "earliest")            # From beginning
    # OR: .option("startingOffsets", "latest")        # Only new messages
    # OR: .option("startingOffsets", '{"orders":{"0":100}}')  # Specific offset
    .option("maxOffsetsPerTrigger", 10000)            # Limit per batch
    .load())

# Kafka message structure:
# key: BINARY (message key)
# value: BINARY (message payload — usually JSON)
# topic: STRING
# partition: INT
# offset: LONG
# timestamp: TIMESTAMP
# timestampType: INT

# Parse the JSON value
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DecimalType

order_schema = StructType([
    StructField("order_id", IntegerType()),
    StructField("customer_id", IntegerType()),
    StructField("amount", DecimalType(12,2)),
    StructField("status", StringType()),
])

parsed_df = (kafka_df
    .select(
        col("key").cast("string").alias("kafka_key"),
        from_json(col("value").cast("string"), order_schema).alias("data"),
        col("topic"),
        col("partition"),
        col("offset"),
        col("timestamp").alias("kafka_timestamp")
    )
    .select("kafka_key", "data.*", "topic", "partition", "offset", "kafka_timestamp"))
```

### Writing to Kafka

```python
# Write processed results back to Kafka
(result_df
    .select(
        col("order_id").cast("string").alias("key"),
        to_json(struct("*")).alias("value")
    )
    .writeStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("topic", "processed-orders")
    .option("checkpointLocation", "/checkpoints/kafka-output")
    .start())
```

In [ ]:
# Kafka message parsing simulation
from pyspark.sql.functions import from_json, col, to_json, struct
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DecimalType, TimestampType
import json

print("📡 KAFKA MESSAGE PARSING SIMULATION")
print("=" * 60)

# Simulate Kafka messages (binary key + JSON value)
kafka_messages = [
    {"key": "order-1001", "value": '{"order_id": 1001, "customer_id": 42, "amount": 99.99, "status": "pending"}', "partition": 0, "offset": 100},
    {"key": "order-1002", "value": '{"order_id": 1002, "customer_id": 17, "amount": 149.50, "status": "pending"}', "partition": 1, "offset": 50},
    {"key": "order-1003", "value": '{"order_id": 1003, "customer_id": 55, "amount": 299.00, "status": "completed"}', "partition": 0, "offset": 101},
    {"key": "order-1001", "value": '{"order_id": 1001, "customer_id": 42, "amount": 99.99, "status": "shipped"}', "partition": 0, "offset": 102},
]

# Create DataFrame simulating Kafka output
kafka_df = spark.createDataFrame(
    [(m["key"], m["value"], m["partition"], m["offset"]) for m in kafka_messages],
    ["key", "value", "partition", "offset"]
)

print("\n📌 Raw Kafka messages (key + value as strings):")
kafka_df.show(truncate=False)

# Parse JSON value
order_schema = StructType([
    StructField("order_id", IntegerType()),
    StructField("customer_id", IntegerType()),
    StructField("amount", DecimalType(12,2)),
    StructField("status", StringType()),
])

parsed = (kafka_df
    .withColumn("data", from_json(col("value"), order_schema))
    .select("key", "data.*", "partition", "offset"))

print("\n📌 Parsed messages (JSON extracted):")
parsed.show(truncate=False)

print("\n💡 In production:")
print("   1. Kafka delivers messages as binary (key + value)")
print("   2. You cast value to STRING, then parse JSON with from_json()")
print("   3. The schema must match the actual JSON structure")
print("   4. Use schema registry (Confluent) for schema management")


---
# 🚀 Section 9: Mini Projects

## Project 1: Streaming Medallion Pipeline

In [ ]:
# Complete streaming medallion: Bronze → Silver → Gold
from pyspark.sql.functions import *

spark.sql("USE iot_manufacturing_dw")

# Bronze: raw stream ingestion
bronze_stream = spark.readStream.format("delta").table("sensor_readings")
bronze_enriched = bronze_stream.withColumn("_ingested_at", current_timestamp())

q1 = (bronze_enriched.writeStream
    .format("delta").outputMode("append")
    .option("checkpointLocation", "/tmp/medallion/bronze_cp")
    .trigger(availableNow=True)
    .toTable("iot_manufacturing_dw.stream_bronze")
)
q1.awaitTermination()

# Silver: cleaned + anomaly flagged
silver_stream = spark.readStream.format("delta").table("stream_bronze")
silver_processed = (silver_stream
    .filter("value IS NOT NULL AND sensor_id IS NOT NULL AND quality_flag = 'valid'")
    .withColumn("is_anomaly", col("value") > 80)
    .withColumn("_processed_at", current_timestamp())
)

q2 = (silver_processed.writeStream
    .format("delta").outputMode("append")
    .option("checkpointLocation", "/tmp/medallion/silver_cp")
    .trigger(availableNow=True)
    .toTable("iot_manufacturing_dw.stream_silver")
)
q2.awaitTermination()

# Gold: windowed aggregates
gold_stream = spark.readStream.format("delta").table("stream_silver")
gold_agg = (gold_stream
    .withWatermark("reading_timestamp", "10 minutes")
    .groupBy(window("reading_timestamp", "1 hour"), "sensor_id")
    .agg(count("*").alias("readings"), round(avg("value"), 2).alias("avg_value"),
         sum(when(col("is_anomaly"), 1).otherwise(0)).alias("anomalies"))
)

q3 = (gold_agg.writeStream
    .format("delta").outputMode("append")
    .option("checkpointLocation", "/tmp/medallion/gold_cp")
    .trigger(availableNow=True)
    .toTable("iot_manufacturing_dw.stream_gold")
)
q3.awaitTermination()

print(f"✅ Streaming Medallion Pipeline:")
print(f"   Bronze: {spark.table('stream_bronze').count():,}")
print(f"   Silver: {spark.table('stream_silver').count():,}")
print(f"   Gold:   {spark.table('stream_gold').count():,}")

## 🎯 Practice: Build a Complete Streaming Pipeline

**Scenario:** Build a real-time order monitoring pipeline that:
1. Reads from a Delta table (simulating a streaming source)
2. Applies a 5-minute tumbling window to count orders per city
3. Detects anomalies (order count > 100 in 5 minutes)
4. Writes results to a Gold table
5. Uses foreachBatch for the anomaly detection

In [ ]:
# ============================================
# 🎯 YOUR TURN: Real-Time Order Monitoring
# ============================================
# Build a streaming pipeline that:
# 1. Reads from techmart_dw.orders as a stream
# 2. Groups by city (from customers) and 5-minute windows
# 3. Counts orders per window
# 4. Flags windows with > 50 orders as "high_volume"
# 5. Writes to a gold table
#
# Hint: Use readStream.format("delta").table() to read Delta as stream
# Hint: Use withWatermark + window for windowed aggregation
#
# Write your streaming pipeline below:


In [ ]:
# ============================================
# ✅ SOLUTION: Real-Time Order Monitoring
# ============================================
from pyspark.sql.functions import col, count, window, when, current_timestamp

print("🚀 REAL-TIME ORDER MONITORING PIPELINE")
print("=" * 60)

# Setup: Create target table
spark.sql("DROP TABLE IF EXISTS techmart_dw.streaming_order_monitor")
spark.sql("""
CREATE TABLE techmart_dw.streaming_order_monitor (
    window_start TIMESTAMP,
    window_end TIMESTAMP,
    order_count BIGINT,
    is_high_volume BOOLEAN,
    _processed_at TIMESTAMP
) USING DELTA
""")

# Read orders as a stream (Delta table as streaming source)
try:
    orders_stream = (spark.readStream
        .format("delta")
        .option("ignoreChanges", "true")
        .table("techmart_dw.orders"))
    
    # Add event time column (use order_timestamp if available, else order_date)
    from pyspark.sql.functions import to_timestamp
    orders_with_time = orders_stream.withColumn(
        "event_time", 
        col("order_timestamp").cast("timestamp")
    )
    
    # Windowed aggregation with watermark
    windowed = (orders_with_time
        .withWatermark("event_time", "10 minutes")
        .groupBy(window("event_time", "5 minutes"))
        .agg(count("*").alias("order_count"))
        .withColumn("window_start", col("window.start"))
        .withColumn("window_end", col("window.end"))
        .withColumn("is_high_volume", col("order_count") > 50)
        .withColumn("_processed_at", current_timestamp())
        .drop("window"))
    
    # Write with foreachBatch for custom logic
    def write_with_alert(batch_df, batch_id):
        batch_df.write.format("delta").mode("append").saveAsTable("techmart_dw.streaming_order_monitor")
        high_vol = batch_df.filter(col("is_high_volume") == True)
        if high_vol.count() > 0:
            print(f"   🚨 ALERT: High volume windows detected in batch {batch_id}!")
            high_vol.show()
    
    (windowed.writeStream
        .foreachBatch(write_with_alert)
        .option("checkpointLocation", "/tmp/order_monitor_checkpoint")
        .trigger(availableNow=True)
        .start()
        .awaitTermination())
    
    print("\n📊 Monitoring results:")
    spark.table("techmart_dw.streaming_order_monitor").orderBy("window_start").show(10)
    
except Exception as e:
    print(f"   ⚠️ Streaming demo: {e}")
    print("   (Requires Delta table with timestamp column)")
    print("   In production, this pattern works seamlessly.")


---
# 🏆 Section 10: Interview Questions

## Q1: Exactly-once semantics in Structured Streaming?

Structured Streaming achieves exactly-once through:
1. **Checkpointing:** Tracks which data has been processed (offset tracking)
2. **Idempotent sinks:** Delta Lake uses transaction log to prevent duplicate writes
3. **Replayable sources:** Kafka/Delta can replay from a specific offset
4. **Atomic commits:** Each micro-batch is an atomic transaction

If a failure occurs, Spark replays from the last checkpoint — no data loss, no duplicates.

## Q2: Watermarks — what and when?

**What:** A threshold that tells Spark how long to wait for late data.
**When:** Required for stateful operations (windowed aggregations, stream-stream joins).

```python
.withWatermark("event_time", "10 minutes")
```
Data arriving > 10 minutes late is dropped. Without watermark, Spark keeps ALL state forever (memory leak).

## Q3: Late-arriving data?

1. **Set watermark:** Define acceptable lateness (e.g., 10 minutes)
2. **Within watermark:** Data is included in the correct window
3. **Beyond watermark:** Data is dropped (or sent to a dead-letter queue)
4. **Batch backfill:** For very late data, run a batch job to reprocess affected windows

## Q4: Stream-stream vs stream-static join?

- **Stream-static:** Stream joined with a regular table (dimension lookup). No watermark needed. Dimension is loaded once.
- **Stream-stream:** Two streams joined. REQUIRES watermarks on both sides (to bound state). More complex but handles two real-time sources.

## Q5: Monitor and recover a failed streaming job?

1. **Monitor:** Check `query.status`, `query.lastProgress`, Spark UI Streaming tab
2. **Detect failure:** Alert on query termination or processing lag
3. **Recover:** Restart from checkpoint (automatic — Spark resumes from last committed offset)
4. **If checkpoint corrupted:** Delete checkpoint, restart from beginning (or specific offset)
5. **Metrics to track:** processing rate, input rate, batch duration, state size

---
# 📗 Section 11: Stateful Streaming

## Stateful vs Stateless Operations

| Type | Operations | State Stored | Memory |
|------|-----------|-------------|--------|
| **Stateless** | filter, select, map | None | Low |
| **Stateful** | groupBy+agg, join, deduplication | Per-key state | High |

Stateful operations maintain state ACROSS micro-batches:

```
Batch 1: [A=10, B=20, A=30]  → State: {A: 40, B: 20}
Batch 2: [A=5, C=15]          → State: {A: 45, B: 20, C: 15}
Batch 3: [B=10, C=5]          → State: {A: 45, B: 30, C: 20}
```

## mapGroupsWithState -- Custom Stateful Logic

```python
from pyspark.sql.streaming.state import GroupState, GroupStateTimeout

def update_session(key, events, state: GroupState):
    if state.hasTimedOut:
        # Session expired -- emit final result
        yield state.get
        state.remove()
    else:
        # Update session state
        current = state.getOption or {"session_id": key, "events": 0, "total": 0}
        for event in events:
            current["events"] += 1
            current["total"] += event.amount
        state.update(current)
        state.setTimeoutDuration(30 * 60 * 1000)  # 30 min timeout

result = (stream
    .groupBy("session_id")
    .mapGroupsWithState(
        outputMode="update",
        timeoutConf=GroupStateTimeout.ProcessingTimeTimeout,
        func=update_session
    ))
```

## Streaming Deduplication

```python
# Deduplicate events within a watermark window
deduped = (stream
    .withWatermark("event_time", "1 hour")
    .dropDuplicates(["event_id", "event_time"]))
```

In [ ]:
# Streaming patterns summary
print("Structured Streaming Quick Reference")
print("=" * 60)

streaming_patterns = {
    "Read from Delta table": """
spark.readStream
    .format("delta")
    .option("ignoreChanges", "true")
    .table("bronze.orders")""",

    "Read from Kafka": """
spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "orders")
    .option("startingOffsets", "latest")
    .load()""",

    "Windowed aggregation": """
(stream
    .withWatermark("event_time", "10 minutes")
    .groupBy(window("event_time", "5 minutes"), "region")
    .agg(count("*").alias("events"), sum("amount").alias("revenue")))""",

    "Write to Delta (append)": """
(result.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "/checkpoints/result")
    .trigger(processingTime="1 minute")
    .toTable("silver.events"))""",

    "foreachBatch with MERGE": """
def upsert_batch(batch_df, batch_id):
    batch_df.createOrReplaceTempView("batch")
    spark.sql(
        "MERGE INTO target t USING batch s ON t.id = s.id "
        "WHEN MATCHED THEN UPDATE SET * "
        "WHEN NOT MATCHED THEN INSERT *"
    )

stream.writeStream.foreachBatch(upsert_batch).start()""",
}

for name, pattern in streaming_patterns.items():
    print(f"\n{name}:")
    print(pattern)


---
# ✅ Notebook Complete!

**What was covered:**
- Streaming concepts: batch vs micro-batch vs real-time, exactly-once
- Structured Streaming: readStream, writeStream, triggers, output modes
- Delta as streaming source and sink
- Windowed aggregations (tumbling windows with watermarks)
- Stream-static joins (dimension enrichment)
- foreachBatch for custom processing logic
- Kafka concepts (conceptual patterns)
- Mini project: complete streaming medallion pipeline (Bronze → Silver → Gold)
- 5 interview questions

**Next:** Notebook 25 — Orchestration & Workflows

---